# 02 - Pressure Intelligence Framework Research & Empirical Distributions

**RR EDGE - Phase 2.1 & Phase 2.1.1: Methodology Research, Validation & Refinement**

This notebook explores the empirical distributions of cricket pressure variables across IPL (2022–2026) and senior Men's T20Is, fits and validates first-innings resource models, formalizes rolling momentum features, investigates bowler vs batter pressure asymmetries, and validates sample size stability.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_all
from src.cricket.competition_config import is_competition_included
from src.cricket.match_state import reconstruct_match_state
from src.cricket.pressure_research import (
    compute_previous_legal_runs,
    compute_advanced_momentum,
    FirstInningsResourceModel,
    classify_batter_pressure_multidimensional,
    classify_bowler_pressure_multidimensional,
    empirical_bayes_shrinkage,
    evaluate_sample_size_confidence,
)

sns.set_theme(style='whitegrid', palette='deep')
%matplotlib inline

## 1. Load Whitelisted Dataset & Reconstruct Pre-Delivery Match State

In [2]:
datasets = load_all()
bbb = datasets['ball_by_ball']
matches = datasets['matches']
comp = datasets['competitions']

# Filter to primary competition whitelist
comp_map = comp.set_index('comp_id')['match_format'].to_dict()
valid_matches = matches[
    matches.apply(
        lambda r: is_competition_included(r['comp_id'], comp_map.get(r['comp_id'], 'T20I')),
        axis=1
    )
].copy()
comp_meta = comp[['comp_id', 'comp_name', 'match_format', 'season']]
valid_matches = valid_matches.merge(comp_meta, on='comp_id', how='left')
valid_matches['comp_type'] = np.where(valid_matches['comp_name'].str.startswith('IPL'), 'IPL', 'T20I')

whitelisted_bbb = bbb[bbb['match_id'].isin(valid_matches['match_id'])].copy()
print(f"Whitelisted matches: {len(valid_matches):,} | Whitelisted deliveries: {len(whitelisted_bbb):,}")

# Reconstruct state with inside-group transforms
state_df = reconstruct_match_state(whitelisted_bbb, valid_matches)
state_df = state_df.merge(valid_matches[['match_id', 'comp_type', 'comp_name', 'match_date']], on='match_id', how='left')
state_df['start_date'] = state_df['match_date']
state_df['prev_6_legal_runs'] = compute_previous_legal_runs(state_df)

# Isolate development cohort (excluding UNCERTAIN_DLS_TIMING)
state_df['runs_required_before'] = state_df['required_runs_before']
dev_df = state_df[state_df['target_certainty'] != 'UNCERTAIN_DLS_TIMING'].copy()
print(f"Primary development cohort deliveries: {len(dev_df):,}")

Whitelisted matches: 853 | Whitelisted deliveries: 194,014
Primary development cohort deliveries: 192,373


## 2. Empirical Distributions — 2nd Innings Chases

In [3]:
chases = dev_df[dev_df['innings'] == 2].copy()
chases['rrr_minus_crr'] = chases['required_run_rate_before'] - chases['current_run_rate_before']

percentiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
print("=== REQUIRED RUN RATE (RRR) PERCENTILES ===")
print(chases['required_run_rate_before'].quantile(percentiles).round(2))

print("\n=== DELTA RRR (RRR - CRR) PERCENTILES ===")
print(chases['rrr_minus_crr'].quantile(percentiles).round(2))

=== REQUIRED RUN RATE (RRR) PERCENTILES ===
0.05     4.20
0.10     5.65
0.25     7.68
0.50     9.66
0.75    12.00
0.90    15.80
0.95    20.82
Name: required_run_rate_before, dtype: float64

=== DELTA RRR (RRR - CRR) PERCENTILES ===
0.05    -6.50
0.10    -4.35
0.25    -1.27
0.50     1.71
0.75     4.97
0.90     8.93
0.95    13.03
Name: rrr_minus_crr, dtype: float64


## 3. Threshold Sensitivity Analysis (Phase 2.1.1 Task 1)

In [4]:
# Empirical distribution check across key proposed thresholds
for r_thresh in [9.0, 10.0, 10.5, 12.0, 14.0, 16.0]:
    mask = chases['required_run_rate_before'] >= r_thresh
    pct = (mask.mean()) * 100
    ipl_pct = (chases[chases['comp_type'] == 'IPL']['required_run_rate_before'] >= r_thresh).mean() * 100
    t20i_pct = (chases[chases['comp_type'] == 'T20I']['required_run_rate_before'] >= r_thresh).mean() * 100
    phase_dist = chases[mask]['innings_phase'].value_counts(normalize=True).round(3).to_dict()
    print(f"RRR >= {r_thresh:>4.1f}: Total={mask.sum():>5d} ({pct:>5.1f}%) | IPL={ipl_pct:>5.1f}% | T20I={t20i_pct:>5.1f}% | Phase: {phase_dist}")

RRR >=  9.0: Total=52999 ( 58.7%) | IPL= 64.2% | T20I= 54.0% | Phase: {'Middle': 0.478, 'Powerplay': 0.287, 'Death': 0.235}
RRR >= 10.0: Total=41514 ( 46.0%) | IPL= 51.1% | T20I= 41.6% | Phase: {'Middle': 0.487, 'Death': 0.279, 'Powerplay': 0.235}
RRR >= 10.5: Total=35979 ( 39.8%) | IPL= 44.1% | T20I= 36.2% | Phase: {'Middle': 0.489, 'Death': 0.309, 'Powerplay': 0.202}
RRR >= 12.0: Total=22731 ( 25.2%) | IPL= 27.0% | T20I= 23.6% | Phase: {'Middle': 0.463, 'Death': 0.44, 'Powerplay': 0.096}
RRR >= 14.0: Total=13296 ( 14.7%) | IPL= 15.4% | T20I= 14.2% | Phase: {'Death': 0.624, 'Middle': 0.351, 'Powerplay': 0.025}
RRR >= 16.0: Total= 8762 (  9.7%) | IPL=  9.9% | T20I=  9.5% | Phase: {'Death': 0.771, 'Middle': 0.224, 'Powerplay': 0.005}


## 4. Advanced Pre-Delivery Momentum Features (Phase 2.1.1 Task 4)

In [5]:
momentum_df = compute_advanced_momentum(chases)
print(momentum_df.describe().round(2))

drought_pct = (momentum_df['prev_6_runs'] <= 3).mean() * 100
surge_pct = (momentum_df['prev_6_runs'] >= 12).mean() * 100
stall_pct = (momentum_df['prev_6_dots'] >= 4).mean() * 100
wkt_pct = (momentum_df['prev_6_wkts'] >= 2).mean() * 100

print(f"\nScoring Drought (<= 3 runs in last 6 balls): {drought_pct:.1f}%")
print(f"Scoring Surge (>= 12 runs in last 6 balls): {surge_pct:.1f}%")
print(f"Dot Pressure Stall (>= 4 dots in last 6 balls): {stall_pct:.1f}%")
print(f"Collapse Spike (>= 2 wickets in last 6 balls): {wkt_pct:.1f}%")

       prev_6_runs  prev_6_dots  prev_6_wkts  prev_over_runs
count     89461.00     89461.00     89461.00        85329.00
mean          8.32         2.19         0.32            8.69
std           4.77         1.39         0.54            4.84
min           0.00         0.00         0.00            0.00
25%           5.00         1.00         0.00            5.00
50%           8.00         2.00         0.00            8.00
75%          11.00         3.00         1.00           12.00
max          36.00         6.00         4.00           32.00

Scoring Drought (<= 3 runs in last 6 balls): 15.4%
Scoring Surge (>= 12 runs in last 6 balls): 23.3%
Dot Pressure Stall (>= 4 dots in last 6 balls): 17.0%
Collapse Spike (>= 2 wickets in last 6 balls): 3.5%


## 5. First-Innings Expected Scoring Model Comparison (Phase 2.1.1 Task 2)

In [6]:
# Chronological 80/20 Match-Level Split
date_col = 'match_date' if 'match_date' in dev_df.columns else 'start_date'
inn1_matches = dev_df[dev_df['innings'] == 1].drop_duplicates('match_id')[['match_id', date_col, 'comp_type']].sort_values(date_col)
n_train = int(len(inn1_matches) * 0.8)
train_mids = set(inn1_matches.iloc[:n_train]['match_id'])
test_mids = set(inn1_matches.iloc[n_train:]['match_id'])

train_inn1 = dev_df[(dev_df['innings'] == 1) & (dev_df['match_id'].isin(train_mids))].copy()
test_inn1 = dev_df[(dev_df['innings'] == 1) & (dev_df['match_id'].isin(test_mids))].copy()

# Fit Model C (Competition-Specific Resource Matrix)
model_c = FirstInningsResourceModel(by_competition=True)
model_c.fit(train_inn1)

print("Sample Expected Remaining Runs from Model C (IPL vs T20I):")
for ov in [0, 6, 12, 16, 18]:
    for wkt in [0, 2, 5]:
        r_ipl = model_c.predict_remaining_runs(ov * 6, wkt, 'IPL')
        r_t20i = model_c.predict_remaining_runs(ov * 6, wkt, 'T20I')
        print(f"  Over {ov:>2d}, {wkt} wkts -> IPL: {r_ipl:>5.1f} runs | T20I: {r_t20i:>5.1f} runs")

Sample Expected Remaining Runs from Model C (IPL vs T20I):
  Over  0, 0 wkts -> IPL: 171.9 runs | T20I: 171.9 runs
  Over  0, 2 wkts -> IPL: 123.7 runs | T20I: 123.7 runs
  Over  0, 5 wkts -> IPL: 170.6 runs | T20I: 170.6 runs
  Over  6, 0 wkts -> IPL: 134.0 runs | T20I: 134.0 runs
  Over  6, 2 wkts -> IPL: 121.9 runs | T20I: 121.9 runs
  Over  6, 5 wkts -> IPL:  84.1 runs | T20I:  84.1 runs
  Over 12, 0 wkts -> IPL:  85.6 runs | T20I:  85.6 runs
  Over 12, 2 wkts -> IPL:  77.8 runs | T20I:  77.8 runs
  Over 12, 5 wkts -> IPL:  63.6 runs | T20I:  63.6 runs
  Over 16, 0 wkts -> IPL:  43.8 runs | T20I:  43.8 runs
  Over 16, 2 wkts -> IPL:  45.6 runs | T20I:  45.6 runs
  Over 16, 5 wkts -> IPL:  37.5 runs | T20I:  37.5 runs
  Over 18, 0 wkts -> IPL:  32.3 runs | T20I:  32.3 runs
  Over 18, 2 wkts -> IPL:  23.9 runs | T20I:  23.9 runs
  Over 18, 5 wkts -> IPL:  19.1 runs | T20I:  19.1 runs


## 6. Bowler vs Batter Asymmetry & Margin Pressure (Phase 2.1.1 Task 3)

In [7]:
# Demonstrating why high RRR != bowler pressure
steep_rrr = chases[chases['required_run_rate_before'] >= 15.0]
tail_closing = steep_rrr[steep_rrr['wickets_remaining_before'] <= 2]
top_order_firepower = steep_rrr[steep_rrr['wickets_remaining_before'] >= 6]

print(f"Total RRR >= 15.0 deliveries: {len(steep_rrr):,}")
print(f"  Batting team 8+ wickets down (Bowler Dominant/Closing Tail): {len(tail_closing):,}")
print(f"  Batting team <= 4 wickets down (Extreme Bowler Pressure): {len(top_order_firepower):,}")

# Acute Margin Pressure
runs_col = 'required_runs_before' if 'required_runs_before' in chases.columns else 'runs_required_before'
margin_crit = chases[(chases['balls_remaining_before'] <= 12) & (chases[runs_col] <= 18)]
print(f"Acute Bowler Margin Pressure (<= 18 runs in <= 12 balls): {len(margin_crit):,} deliveries")


Total RRR >= 15.0 deliveries: 10,631
  Batting team 8+ wickets down (Bowler Dominant/Closing Tail): 3,108
  Batting team <= 4 wickets down (Extreme Bowler Pressure): 1,896
Acute Bowler Margin Pressure (<= 18 runs in <= 12 balls): 2,232 deliveries


## 7. Sample Size Stability & Empirical Bayes Shrinkage (Phase 2.1.1 Task 6)

In [8]:
# Shrinkage demonstration
print("Empirical Bayes Shrinkage Simulation (Prior SR = 130.0, M = 30 balls):")
for n_balls in [6, 12, 24, 60, 120, 250]:
    shrunk_sr = empirical_bayes_shrinkage(200.0, n_balls, 130.0, prior_weight=30.0)
    conf = evaluate_sample_size_confidence(n_balls)
    print(f"  n={n_balls:>3d} balls (Observed SR=200.0) -> Shrunk SR: {shrunk_sr:>5.1f} | Tier: {conf['status']:<12s}")

Empirical Bayes Shrinkage Simulation (Prior SR = 130.0, M = 30 balls):
  n=  6 balls (Observed SR=200.0) -> Shrunk SR: 141.7 | Tier: INSUFFICIENT
  n= 12 balls (Observed SR=200.0) -> Shrunk SR: 150.0 | Tier: INSUFFICIENT
  n= 24 balls (Observed SR=200.0) -> Shrunk SR: 161.1 | Tier: PROVISIONAL 
  n= 60 balls (Observed SR=200.0) -> Shrunk SR: 176.7 | Tier: CONFIDENT   
  n=120 balls (Observed SR=200.0) -> Shrunk SR: 186.0 | Tier: CONFIDENT   
  n=250 balls (Observed SR=200.0) -> Shrunk SR: 192.5 | Tier: CONFIDENT   
